In [ ]:
import argparse
import colorsys
import gc
import getpass
import hashlib
import json
import os
import re
import time
from datetime import datetime
from pathlib import Path
from types import SimpleNamespace
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import requests
from PIL import Image, ImageDraw, ImageFont
from rasterio.transform import GCPTransformer
from scipy.spatial import cKDTree
from shapely import wkt

Image.MAX_IMAGE_PIXELS = None

from PIL import ImageFont
from PIL import Image, ImageDraw
from IPython.display import clear_output
import trino
from trino import exceptions as trino_exceptions
from trino.auth import OAuth2Authentication


# Configuration and Processing Parameters

Define input/output directories, study-area settings, acquisition filters, and processing options.


In [ ]:
# =============================================================================
# USER CONFIGURATION
# =============================================================================

# Root directory containing all project data.
# Set this to your local project folder.
ROOT = Path.cwd()

# Output directory for generated images and results.
OUTPUT_DIR = ROOT / "Images"

# Directory containing downloaded Sentinel-1 scenes.
# Example:
# ROOT/
# ├── S1_downloads/
# │   └── 2025/
# └── Images/
DOWNLOAD_DIR = ROOT / "S1_downloads" / "2025"

# Analysis date range (inclusive)
DATE_FROM = "2025-01-01"
DATE_TO_INCLUSIVE = "2026-02-28"

# Search margin(s) around aircraft positions in kilometres.
MARGINS_KM = [5]

# Limit the number of Sentinel-1 scenes processed.
# Set to None to process all available scenes.
LIMIT_SCENES = None

# =============================================================================
# DATA SOURCE OPTIONS
# =============================================================================

# If True, use only locally stored data.
LOCAL_ONLY = False

# If True, disable all network requests.
OFFLINE = False

# If True, refresh the Sentinel scene catalogue.
REFRESH_CATALOGUE = False

# =============================================================================
# STUDY AREA CONFIGURATION
# =============================================================================

# Bounding box:
# (min_longitude, min_latitude, max_longitude, max_latitude)
BBOX = (
    -10.31,
    54.169,
    -10.002,
    54.349,
)

# Sentinel-1 orbit direction to analyse.
ORBIT_DIRECTION = "DESCENDING"

# Area of interest polygon in WKT format. Default set to North-west Ireland.
polygon_wkt = (
    "POLYGON(("
    "-7.590541 54.824537,"
    "-11.678641 55.245177,"
    "-12.069724 53.751265,"
    "-8.125319 53.334224,"
    "-7.590541 54.824537"
    "))"
)

# Additional buffer applied to the AOI (degrees).
buffer = 0.1

# Platform inclination angle (degrees).
inclination = 81.82

# =============================================================================
# PROCESSING OPTIONS
# =============================================================================

# Skip scenes without usable aircraft tracks.
SAVE_SCENES_WITHOUT_AIRCRAFT = False

# OpenSky Authentication and Data Access

Configure OpenSky authentication, Sentinel-1 catalogue access, and local caching utilities.

In [ ]:
# OpenSky Network account credentials
# Create a free account at https://opensky-network.org
OPENSKY_USER = "Enter your opensky username/email here"

# Height-Induced Range Correction

Compute the displacement of aircraft signatures in Sentinel-1 range coordinates using aircraft altitude, local incidence angle, and acquisition geometry derived from Sentinel-1 annotation metadata.

In [ ]:
class RangeHeightCorrection:
    """
    Computes aircraft height-induced range displacement in Sentinel-1 GRD imagery
    using the nearest geolocation grid point and the local incidence angle.

    Parameters
    ----------
    xml_path : str or Path
        Path to the Sentinel-1 annotation XML file.

    Notes
    -----
    The correction assumes:

    - Aircraft altitude is provided in metres.
    - Sentinel-1 geolocation grid values are used without interpolation.
    - The range-axis orientation is determined directly from the Sentinel-1
      annotation metadata.
    """

    def __init__(self, xml_path):

        # Read Sentinel-1 annotation metadata
        root = ET.parse(xml_path).getroot()

        # Native GRD pixel spacing (metres)
        self.range_spacing = float(
            root.findtext('.//imageInformation/rangePixelSpacing')
        )
        self.azimuth_spacing = float(
            root.findtext('.//imageInformation/azimuthPixelSpacing')
        )

        if (
            not np.all(np.isfinite([self.range_spacing, self.azimuth_spacing]))
            or min(self.range_spacing, self.azimuth_spacing) <= 0
        ):
            raise ValueError(
                "Invalid pixel spacing in Sentinel-1 annotation."
            )

        # Load geolocation grid points
        points = root.findall('.//geolocationGridPoint')

        # Latitude and longitude of geolocation grid points
        self.coordinates = np.array([
            [
                float(p.findtext('latitude')),
                float(p.findtext('longitude'))
            ]
            for p in points
        ])

        # Local incidence angle (degrees)
        self.angles = np.array([
            float(p.findtext('incidenceAngle'))
            for p in points
        ])

        # Slant-range time (seconds)
        self.times = np.array([
            float(p.findtext('slantRangeTime'))
            for p in points
        ])

        if (
            not len(points)
            or not np.all(np.isfinite(self.coordinates))
            or not np.all(np.isfinite(self.times))
        ):
            raise ValueError(
                "Missing or invalid geolocation grid."
            )

        # Fast nearest-neighbour geographic lookup
        self.tree = cKDTree(self.coordinates)

        # Determine native range-axis orientation directly from the
        # Sentinel-1 annotation metadata.
        lines = {}

        for p in points:
            lines.setdefault(
                int(p.findtext('line')),
                []
            ).append(
                (
                    float(p.findtext('pixel')),
                    float(p.findtext('slantRangeTime'))
                )
            )

        signs = []

        for entries in lines.values():
            ordered = np.array(sorted(entries))

            if len(ordered) > 1:
                slopes = (
                    np.diff(ordered[:, 1]) /
                    np.diff(ordered[:, 0])
                )
                signs.extend(np.sign(slopes).tolist())

        if not signs or 0 in signs or len(set(signs)) != 1:
            raise ValueError(
                "Cannot establish a consistent range-column direction."
            )

        self.range_direction = signs[0]

    def correct_range(self, lon, lat, height_m):
        """
        Compute the radar range displacement caused by target height.

        Parameters
        ----------
        lon : float
            Longitude in decimal degrees.

        lat : float
            Latitude in decimal degrees.

        height_m : float
            Aircraft altitude in metres.

        Returns
        -------
        dict
            Dictionary containing:

            incidence_angle
                Local incidence angle (degrees).

            slant_range_time
                Slant-range time (seconds).

            R_slant_m
                Slant-range distance (metres).

            height_range_shift_m
                Height-induced range displacement (metres).

            height_range_shift_px
                Height-induced range displacement (pixels).
        """

        if (
            not np.all(np.isfinite([lon, lat, height_m]))
            or height_m < 0
        ):
            raise ValueError(
                "Missing or invalid aircraft altitude or coordinates."
            )

        _, index = self.tree.query([lat, lon])

        theta_deg = float(self.angles[index])

        if not np.isfinite(theta_deg) or not 0 < theta_deg < 90:
            raise ValueError(
                "Incidence angle must be between 0 and 90 degrees."
            )

        # Height-induced displacement:
        # ΔR = h / tan(θ)
        shift_m = float(
            height_m / np.tan(np.deg2rad(theta_deg))
        )

        return {
            "incidence_angle": theta_deg,
            "slant_range_time": float(self.times[index]),
            "R_slant_m": float(
                self.times[index] * 299792458.0 / 2.0
            ),
            "height_range_shift_m": shift_m,
            "height_range_shift_px":
                -self.range_direction * shift_m / self.range_spacing,
        }

# Aircraft Track Processing and Trajectory Interpolation

Retrieve OpenSky aircraft observations, interpolate trajectories over the Sentinel-1 acquisition interval, and apply Doppler and height-induced geometric corrections.

In [ ]:
def scene_times_from_tiff_name(tiff_path):
    # Extract acquisition start and end times from the Sentinel-1 filename.
    match = re.search(r"-(\d{8}t\d{6})-(\d{8}t\d{6})-", tiff_path.name.lower())

    if not match:
        raise ValueError(f"Could not parse scene time from {tiff_path.name}")

    start = datetime.strptime(match.group(1), "%Y%m%dt%H%M%S")
    stop = datetime.strptime(match.group(2), "%Y%m%dt%H%M%S")

    return (
        start.strftime("%Y-%m-%dT%H:%M:%SZ"),
        stop.strftime("%Y-%m-%dT%H:%M:%SZ"),
        start.strftime("%Y-%m-%d")
    )


def prepare_aircraft_dataframe(cur, t_start, t_stop, xml_path):
    # Query OpenSky data and prepare interpolated aircraft tracks
    # for the current Sentinel-1 acquisition.

    rows = query_opensky(
        cur,
        t_begin_str=t_start,
        t_end_str=t_stop,
        polygon_wkt=polygon_wkt,
        buffer_deg=buffer,
        limit=10000
    )

    # Skip further processing if no aircraft observations are found.
    if len(rows) == 0:
        print("No OpenSky rows found in time/polygon window.")
        return pd.DataFrame()

    colnames = [desc[0] for desc in cur.description]
    df = pd.DataFrame(rows, columns=colnames)

    if df.empty:
        return df

    v_sat = 7400  # Satellite velocity used in the Doppler displacement approximation.

    geometry = RangeHeightCorrection(xml_path)
    azimuth_pixel_spacing = geometry.azimuth_spacing

    # OpenSky geoaltitude is used as the aircraft altitude.
    # Barometric altitude is not substituted.
    if "geoaltitude" not in df.columns:
        df["geoaltitude"] = np.nan

    for column in ["lat", "lon", "velocity", "heading", "geoaltitude"]:
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce"
        ).replace([np.inf, -np.inf], np.nan)

    # Select only airborne aircraft observations.
    df_place_look = df[df["onground"] == False].copy()

    if df_place_look.empty:
        return df_place_look

    t_sar_start = pd.to_datetime(t_start, utc=True)
    t_sar_end = pd.to_datetime(t_stop, utc=True)

    df_place_look["time"] = pd.to_datetime(
        df_place_look["time"],
        unit="s",
        utc=True
    )

    df_place_look = df_place_look.drop_duplicates(
        subset=["icao24", "time"],
        keep="first"
    )

    df_place_look = df_place_look.sort_values(["icao24", "time"])

    all_planes = []

    # Interpolate aircraft positions and kinematic parameters
    # to the Sentinel-1 acquisition interval.
    for icao24, grp in df_place_look.groupby("icao24"):
        grp = grp.copy().sort_values("time")
        grp = grp.set_index("time")

        for t in [t_sar_start, t_sar_end]:
            if t not in grp.index:
                grp.loc[t] = np.nan

        grp = grp.sort_index()

        grp["callsign"] = grp["callsign"].ffill().bfill()
        grp["icao24"] = icao24

        grp["heading_rad"] = np.deg2rad(grp["heading"])
        grp["heading_sin"] = np.sin(grp["heading_rad"])
        grp["heading_cos"] = np.cos(grp["heading_rad"])

        cols = [
            "lat",
            "lon",
            "velocity",
            "heading_sin",
            "heading_cos",
            "geoaltitude"
        ]

        grp[cols] = grp[cols].interpolate(
            method="time",
            limit_area="inside"
        )

        grp["heading"] = np.rad2deg(
            np.arctan2(grp["heading_sin"], grp["heading_cos"])
        )

        grp["heading"] = (grp["heading"] + 360) % 360

        grp = grp.reset_index()
        all_planes.append(grp)

    if len(all_planes) == 0:
        return pd.DataFrame()

    df_place_look_interp = pd.concat(all_planes, ignore_index=True)
    df_place_look_interp = df_place_look_interp.copy()

    gdf = gpd.GeoDataFrame(
        df_place_look_interp,
        geometry=gpd.points_from_xy(
            df_place_look_interp["lon"],
            df_place_look_interp["lat"]
        ),
        crs="EPSG:4326"
    )

    polygon = wkt.loads(polygon_wkt)

    df_filtered = gdf[
        (gdf["time"] >= t_sar_start) &
        (gdf["time"] <= t_sar_end) &
        (gdf.within(polygon))
    ].copy()

    if df_filtered.empty:
        return df_filtered

    # Retain only the start and end points of each interpolated aircraft track.
    df_filtered = df_filtered.sort_values(["icao24", "time"]).copy()

    track_rows = []

    for _, grp in df_filtered.groupby("icao24", sort=False):
        grp = grp.sort_values("time")

        if len(grp) >= 2:
            track_rows.append(grp.iloc[[0, -1]])

    if not track_rows:
        return pd.DataFrame()

    df_filtered = pd.concat(track_rows, ignore_index=True)

    # Apply the height-induced range correction to each track endpoint.
    # Complete start/end pairs are required for track processing.
    projected_tracks = []

    for icao24, grp in df_filtered.groupby("icao24", sort=False):

        required = grp[
            ["lon", "lat", "geoaltitude", "velocity", "heading"]
        ].to_numpy(dtype=float)

        if not np.all(np.isfinite(required)):
            print(
                f"Skipping {icao24}: missing geometric altitude, "
                f"position, or velocity at track endpoint."
            )
            continue

        try:
            projections = [
                geometry.correct_range(
                    p.lon,
                    p.lat,
                    p.geoaltitude
                )
                for p in grp.itertuples()
            ]

        except ValueError as err:
            print(f"Skipping {icao24}: {err}")
            continue

        grp = grp.copy()

        for key in projections[0]:
            grp[key] = [p[key] for p in projections]

        projected_tracks.append(grp)

    if not projected_tracks:
        return pd.DataFrame()

    df_filtered = pd.concat(projected_tracks, ignore_index=True)

    def offset_doppler1(
        sog,
        cog,
        R_slant,
        lat_aircraft,
        incidence_angle,
        v_sat,
        inclination_angle
    ):
        """
        Estimate azimuth displacement caused by aircraft motion
        relative to the radar line of sight.
        """

        lat_aircraft = np.deg2rad(lat_aircraft)
        cog = np.deg2rad(cog)
        incidence_angle = np.deg2rad(incidence_angle)
        inclination_angle = np.deg2rad(inclination_angle)

        cog_AIS_proj = cog - np.arcsin(
            np.clip(
                np.cos(inclination_angle) / np.cos(lat_aircraft),
                -1.0,
                1.0
            )
        )

        return (
            sog
            * np.sin(cog_AIS_proj)
            * R_slant
            * np.sin(incidence_angle)
            / v_sat
        )

    doppler_offset = offset_doppler1(
        sog=df_filtered["velocity"].values,
        cog=df_filtered["heading"].values,
        R_slant=df_filtered["R_slant_m"].values,
        lat_aircraft=df_filtered["lat"].values,
        incidence_angle=df_filtered["incidence_angle"].values,
        v_sat=v_sat,
        inclination_angle=inclination
    )

    df_filtered["doppler_offset_m"] = doppler_offset
    df_filtered["doppler_offset_px"] = (
        df_filtered["doppler_offset_m"] / azimuth_pixel_spacing
    )

    return df_filtered

# Sentinel-1 and OpenSky Data Access

Discover Sentinel-1 scenes, manage downloads, authenticate with OpenSky, and cache data required for aircraft-track processing.

In [ ]:
CATALOGUE_URL = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"

DOWNLOAD_URL = "https://download.dataspace.copernicus.eu/odata/v1"

TOKEN_URL = (
    "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/"
    "protocol/openid-connect/token"
)
_QUERY_CLIENT = None
_SCALING = {}
_OPENSKY_AUTHENTICATIONS = {}
DEFAULT_MARGINS_KM = (5.0,)
def write_json(path, record):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.part')
    temporary.write_text(json.dumps(record, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
    temporary.replace(path)

def public_session():
    from requests.adapters import HTTPAdapter
    from urllib3.util.retry import Retry
    session = requests.Session()
    retry = Retry(total=4, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504], allowed_methods=['GET'])
    session.mount('https://', HTTPAdapter(max_retries=retry))
    return session

def scene_is_valid(xml, tiff):
    try:
        root = ET.parse(xml).getroot()
        if root.findtext('.//adsHeader/productType') != 'GRD':
            return False
        with rasterio.open(tiff) as src:
            if not src.gcps[0] or src.dtypes[0] != 'uint16':
                return False
            info = root.find('.//imageInformation')
            if (src.width, src.height) != (int(info.findtext('numberOfSamples')), int(info.findtext('numberOfLines'))):
                return False
            from rasterio.windows import Window
            src.read(1, window=Window(0, 0, 1, 1))
            src.read(1, window=Window(src.width-1, src.height-1, 1, 1))
        return True
    except (OSError, ValueError, TypeError, ET.ParseError, rasterio.errors.RasterioError):
        return False

def scan_local(root, excluded=(), validate=False):
    """Index ONLY the supplied batch directory. Default: no file-content reads.

    Directory listings and file lengths do not intentionally hydrate OneDrive
    TIFF placeholders. Content is checked later, for the scene being rendered.
    """
    excluded = {Path(p).resolve() for p in excluded}
    products = {}
    if not root.is_dir():
        return products
    for folder, dirs, files in os.walk(root, followlinks=False):
        dirs[:] = [d for d in dirs if d not in ('__pycache__', '.git') and (Path(folder)/d).resolve() not in excluded]
        parent = Path(folder)
        safe = next((p.name for p in (parent, *parent.parents) if p.name.upper().endswith('.SAFE')), None)
        if not safe or not re.match(r'S1[A-Z]_IW_GRDH_1S', safe):
            continue
        entry = products.setdefault(safe, {'files': {}})
        for name in files:
            if '-vv-' in name.lower() and Path(name).suffix.lower() in ('.xml', '.tif', '.tiff'):
                entry['files'].setdefault(name.lower(), []).append(parent/name)
    for entry in products.values():
        entry['pair'] = find_valid_pair(entry['files'],validate=validate)
    return products

def find_valid_pair(files, validate=False):
    for name, paths in files.items():
        if not name.endswith(('.tif', '.tiff')):
            continue
        for xml in files.get(Path(name).stem+'.xml', []):
            for tiff in paths:
                present=xml.is_file() and tiff.is_file() and xml.stat().st_size>0 and tiff.stat().st_size>0
                if present and (not validate or scene_is_valid(xml,tiff)):
                    return {'xml': str(xml), 'tiff': str(tiff)}
    return None

def catalogue_products(session, begin, stop):
    """Public metadata; full pagination and an exclusive end time."""
    x0,y0,x1,y1 = BBOX
    region = f'POLYGON(({x0} {y0},{x1} {y0},{x1} {y1},{x0} {y1},{x0} {y0}))'
    query = (
        "Collection/Name eq 'SENTINEL-1' and "
        f"OData.CSC.Intersects(area=geography'SRID=4326;{region}') and "
        f"ContentDate/Start ge {begin.strftime('%Y-%m-%dT00:00:00Z')} and "
        f"ContentDate/Start lt {stop.strftime('%Y-%m-%dT00:00:00Z')} and "
        "Attributes/OData.CSC.StringAttribute/any(a: a/Name eq 'orbitDirection' and "
        f"a/OData.CSC.StringAttribute/Value eq '{ORBIT_DIRECTION}') and "
        "Attributes/OData.CSC.StringAttribute/any(a: a/Name eq 'productType' and "
        "a/OData.CSC.StringAttribute/Value eq 'IW_GRDH_1S')")
    url = CATALOGUE_URL
    params = {'$filter': query, '$orderby': 'ContentDate/Start asc', '$top': 1000}
    products = []
    seen_pages = set()
    while url:
        if url in seen_pages:
            raise RuntimeError('Catalogue pagination repeated a page.')
        seen_pages.add(url)
        response = session.get(url, params=params, timeout=(20, 120))
        response.raise_for_status()
        page = response.json()
        products.extend(page['value'])
        url = page.get('@odata.nextLink')
        if url and not url.startswith(CATALOGUE_URL):
            raise RuntimeError('Unexpected catalogue pagination host.')
        params = None
    return sorted({p['Id']: p for p in products if '_1SDV_' in p['Name'] or '_1SSV_' in p['Name']}.values(), key=lambda p:p['ContentDate']['Start'])

def local_products(index, begin, stop):
    selected = []
    for name, entry in index.items():
        if not entry['pair']:
            continue
        root = ET.parse(entry['pair']['xml']).getroot()
        info = root.find('.//imageInformation')
        t = pd.to_datetime(info.findtext('productFirstLineUtcTime'), utc=True)
        if not begin <= t < stop or root.findtext('.//pass', '').upper() != ORBIT_DIRECTION:
            continue
        points = root.findall('.//geolocationGridPoint')
        lons = [float(p.findtext('longitude')) for p in points]
        lats = [float(p.findtext('latitude')) for p in points]
        # Local scene discovery uses a bounding-box intersection.
        # Catalogue queries use the full Sentinel-1 scene footprint.
        if max(lons) < BBOX[0] or min(lons) > BBOX[2] or max(lats) < BBOX[1] or min(lats) > BBOX[3]:
            continue
        selected.append({'Name': name, 'Id': None, 'ContentDate': {'Start': t.isoformat()}})
    return sorted(selected, key=lambda p:p['ContentDate']['Start'])

class CopernicusDownloads:
    def __init__(self, session, offline=False):
        self.session = session
        self.offline = offline
        self.access = os.getenv('CDSE_ACCESS_TOKEN')
        self.refresh = None
        self.expires = time.monotonic()+240 if self.access else 0

    def token(self, force=False):
        if self.offline:
            raise RuntimeError('Offline: the requested Sentinel file is not available locally.')
        if self.access and not force and time.monotonic() < self.expires:
            return self.access
        if self.refresh:
            data = {'client_id':'cdse-public', 'grant_type':'refresh_token', 'refresh_token':self.refresh}
            reply = self.session.post(TOKEN_URL, data=data, timeout=(20, 60))
            if reply.ok:
                record = reply.json()
                self._set_token(record)
                return self.access
            self.refresh = None
        user = os.getenv('CDSE_USERNAME') or input('Copernicus Data Space e-mail: ').strip()
        password = os.getenv('CDSE_PASSWORD') or getpass.getpass('Copernicus password (hidden): ')
        data = {'client_id':'cdse-public','grant_type':'password','username':user,'password':password}
        totp = os.getenv('CDSE_TOTP')
        if totp:
            data['totp'] = totp
        reply = self.session.post(TOKEN_URL, data=data, timeout=(20,60))
        if not reply.ok:
            raise RuntimeError(f'Copernicus login failed (HTTP {reply.status_code}); check account/2FA. Credentials are not stored.')
        self._set_token(reply.json())
        return self.access

    def _set_token(self, record):
        self.access = record['access_token']
        self.refresh = record.get('refresh_token', self.refresh)
        self.expires = time.monotonic()+max(1, float(record.get('expires_in',300))-30)

    def get(self, url, authenticated=False, **kwargs):
        if self.offline:
            raise RuntimeError('Offline: a required remote file or listing is missing.')
        headers = {'Authorization':'Bearer '+self.token()} if authenticated else {}
        reply = self.session.get(url, headers=headers, timeout=(20,180), **kwargs)
        if reply.status_code == 401:
            reply.close()
            headers = {'Authorization':'Bearer '+self.token(force=authenticated)}
            reply = self.session.get(url, headers=headers, timeout=(20,180), **kwargs)
        reply.raise_for_status()
        return reply

    def nodes(self, base):
        with self.get(base+'/Nodes') as response:
            payload = response.json()
        return payload.get('value', payload.get('result', []))

    def download(self, url, path, expected=0):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        if path.is_file() and path.stat().st_size > 0 and (not expected or path.stat().st_size==expected):
            return path
        part = path.with_name(path.name+'.part')
        print('Downloading missing VV file:',path.name,flush=True)
        with self.get(url, authenticated=True, stream=True) as response:
            size = 0
            with part.open('wb') as handle:
                for chunk in response.iter_content(1024*1024):
                    if chunk:
                        handle.write(chunk)
                        size += len(chunk)
            content_length = int(response.headers.get('Content-Length',0))
            if size==0 or (expected and size!=expected) or (content_length and not response.headers.get('Content-Encoding') and size!=content_length):
                raise RuntimeError('Incomplete download: '+path.name)
        part.replace(path)
        return path

    def ensure_scene(self, product, index, download_dir):
        name = product['Name']
        known = index.get(name, {'files':{},'pair':None})
        if known['pair'] and scene_is_valid(**known['pair']):
            print('Reusing local scene:',name,flush=True)
            return known['pair']
        if not product.get('Id'):
            raise RuntimeError('Missing local scene; catalogue ID needed for download: '+name)
        product_base = f"{DOWNLOAD_URL}/Products({product['Id']})/Nodes({name})"
        annotations = self.nodes(product_base+'/Nodes(annotation)')
        measurements = self.nodes(product_base+'/Nodes(measurement)')
        xmls = {Path(n['Name']).stem.lower(): n for n in annotations if '-vv-' in n['Name'].lower() and n['Name'].lower().endswith('.xml')}
        pairs = [(n,xmls[Path(n['Name']).stem.lower()]) for n in measurements
                 if '-vv-' in n['Name'].lower() and n['Name'].lower().endswith(('.tif','.tiff')) and Path(n['Name']).stem.lower() in xmls]
        if len(pairs)!=1:
            raise RuntimeError(f'{name}: expected one matching VV XML/TIFF pair; got {len(pairs)}.')
        measurement,annotation = pairs[0]
        selected = {}
        for kind,node,folder in [('xml',annotation,'annotation'),('tiff',measurement,'measurement')]:
            file_name = node['Name']
            # File names are API data, not local paths.
            if Path(file_name).name!=file_name or '/' in file_name or '\\' in file_name:
                raise RuntimeError('Unexpected remote file name.')
            candidates = known['files'].get(file_name.lower(), [])
            expected = int(node.get('ContentLength',0) or 0)
            existing = next((p for p in candidates if p.is_file() and p.stat().st_size>0 and (not expected or p.stat().st_size==expected)),None)
            selected[kind] = str(existing or self.download(product_base+f'/Nodes({folder})/Nodes({file_name})/$value', download_dir/name/file_name,expected))
        if not scene_is_valid(**selected):
            raise RuntimeError('Downloaded/reused XML/TIFF failed the scene check: '+name)
        index[name]={'pair':selected,'files':known['files']}
        return selected

def opensky_browser_login(url):
    """Pause before token polling so browser login can finish in Jupyter or CLI."""

    import webbrowser

    print(
        "\nComplete the OpenSky login in your web browser.",
        flush=True
    )

    print(
        "Open the following link if the browser does not open automatically:\n"
        + url,
        flush=True
    )

    try:
        webbrowser.open(url, new=2)
    except Exception:
        pass

    input(
        "After completing the login, return here and press ENTER: "
    )

def opensky_authentication(user):
    """Reuse OAuth authentication across multiple workflow executions."""
    import trino.auth
    bearer=getattr(trino.auth,'_OAuth2TokenBearer',None)
    if bearer is not None and hasattr(bearer,'MAX_OAUTH_ATTEMPTS'):
        bearer.MAX_OAUTH_ATTEMPTS=max(bearer.MAX_OAUTH_ATTEMPTS,60)
    if user not in _OPENSKY_AUTHENTICATIONS:
        _OPENSKY_AUTHENTICATIONS[user]=trino.auth.OAuth2Authentication(opensky_browser_login)
    return _OPENSKY_AUTHENTICATIONS[user]

class OpenSkyCache:
    FIELDS = ('time','icao24','callsign','lat','lon','velocity','heading','geoaltitude','baroaltitude','onground')
    def __init__(self, cache_dir, user='', offline=False):
        self.path = Path(cache_dir)
        self.user = user
        self.offline = offline
        self.connection = None
        self.live = None

    def _cursor(self):
        if self.live is None:
            import trino
            user = (
                    self.user
                    or os.getenv("OPENSKY_USER")
                    or input(
                        "OpenSky username "
                        "(must match the account used in the browser login): "
                    ).strip()
                )
            user = user.strip().lower()
            if not user:
                raise RuntimeError('OpenSky username is required.')
            self.user=user
            self.connection = trino.dbapi.connect(host='trino.opensky-network.org',port=443,user=user,http_scheme='https',auth=opensky_authentication(user),catalog='minio',schema='osky',request_timeout=120)
            self.live = self.connection.cursor()
        return self.live

    def authenticate(self):
        if self.offline:
            raise RuntimeError('Offline mode does not perform browser login.')
        try:
            live=self._cursor()
            live.execute('SELECT 1')
            rows=live.fetchall()
            if not rows or rows[0][0]!=1:
                raise RuntimeError('Unexpected OpenSky login-check response.')
        except BaseException:
            self.close()
            raise
        print("OpenSky login successful.", flush=True)
        return self.user

    def query(self, cursor, t_begin_str, t_end_str, polygon_wkt, buffer_deg=0.1, limit=None):
        begin = int(pd.to_datetime(t_begin_str,utc=True).timestamp())-300
        end = int(pd.to_datetime(t_end_str,utc=True).timestamp())+300
        sql = f"""SELECT {', '.join(self.FIELDS)} FROM state_vectors_data4
            WHERE hour BETWEEN {begin-begin%3600} AND {end-end%3600}
              AND time BETWEEN {begin} AND {end}
              AND onground = false
              AND ST_Contains(ST_Buffer(ST_GeometryFromText('{polygon_wkt}'), {buffer_deg}), ST_Point(lon, lat))
            ORDER BY icao24, time"""
        key = hashlib.sha256(sql.encode()).hexdigest()
        cached = self.path/f'adsb_{key}.json'
        if cached.exists():
            record = json.loads(cached.read_text(encoding='utf-8'))
            if record.get('sql')!=sql or tuple(record['columns'])!=self.FIELDS:
                raise RuntimeError('ADS-B cache does not match query/schema.')
            print('Using ADS-B cache:',len(record['rows']),'rows',flush=True)
        else:
            if self.offline:
                raise RuntimeError('Offline: no complete ADS-B cache for '+t_begin_str)
            print('Fetching all ADS-B aircraft:',t_begin_str,flush=True)
            live=self._cursor()
            live.execute(sql)
            rows=live.fetchall()
            record={'columns':[c[0] for c in live.description],'rows':rows,'sql':sql}
            if tuple(record['columns'])!=self.FIELDS:
                raise RuntimeError('OpenSky returned an unexpected schema.')
            write_json(cached,record)
        cursor.description=[(c,) for c in record['columns']]
        return record['rows']

    def close(self):
        if self.live is not None:self.live.close()
        if self.connection is not None:self.connection.close()
        self.live=None
        self.connection=None

def login_opensky(user=""):
    """Authenticate with OpenSky before processing data."""
    client=OpenSkyCache(Path.cwd(),user=user)
    try:
        return client.authenticate()
    finally:
        client.close()

def query_opensky(cursor, **kwargs):
    if _QUERY_CLIENT is None:
        raise RuntimeError('ADS-B client not initialized.')
    return _QUERY_CLIENT.query(cursor,**kwargs)


# SAR Image Processing and Visualisation

Load Sentinel-1 imagery, apply intensity scaling, project aircraft tracks into SAR image coordinates, and generate annotated visualisations of candidate aircraft signatures.

In [ ]:
def percentile_from_histogram(hist, percentiles):
    cumulative = np.cumsum(hist)
    ranks = (cumulative[-1] - 1) * np.asarray(percentiles) / 100.0
    low, high = (np.floor(ranks).astype(int), np.ceil(ranks).astype(int))
    a = np.searchsorted(cumulative, low, side='right')
    b = np.searchsorted(cumulative, high, side='right')
    return a + (b - a) * (ranks - low)

def load_period_image(tiff_path):
    """
    Load the original Sentinel-1 uint16 image and apply a percentile-based
    contrast stretch using the 2nd and 98th percentiles.

    A lookup table is cached to avoid recomputing the histogram when
    processing the same image multiple times.

    No image rotation or reflection is applied.
    """

    from rasterio.windows import Window
    stat = Path(tiff_path).stat()
    key=(str(Path(tiff_path).resolve()),stat.st_size,stat.st_mtime_ns)
    with rasterio.open(tiff_path) as src:
        if src.dtypes[0]!='uint16':raise ValueError('Expected original uint16 GRD.')
        if key not in _SCALING:
            hist=np.zeros(65536,dtype=np.int64)
            for top in range(0,src.height,128):
                strip=src.read(1,window=Window(0,top,src.width,min(128,src.height-top)))
                hist+=np.bincount(strip.ravel(),minlength=65536)
            low,high=percentile_from_histogram(hist,[2,98])
            lut=(np.clip((np.arange(65536,dtype=float)-low)/(high-low)*255,0,255).astype(np.uint8) if high>low else np.zeros(65536,dtype=np.uint8))
            _SCALING[key]={'lut':lut,'low':float(low),'high':float(high)}
        scale=_SCALING[key]
        image=Image.new('RGB',(src.width,src.height))
        try:
            for top in range(0,src.height,128):
                strip=src.read(1,window=Window(0,top,src.width,min(128,src.height-top)))
                tile=Image.fromarray(scale['lut'][strip]).convert('RGB')
                image.paste(tile,(0,top));tile.close()
            transform=GCPTransformer(src.gcps[0])
        except BaseException:
            image.close();raise
    return image,transform,{'percentile_2':scale['low'],'percentile_98':scale['high']}

def draw_sar_image(tiff_path, df_filtered, output_jpg, margin_km=5.0):
    # Visualization parameters
    POINT_SIZE = 30
    DOPPLER_WIDTH = float(margin_km) * 100.0
    OUTLINE_WIDTH = 10
    HEADING_LEN = 200
    TEXT_DIS = 80
    FONT_SIZE = 80

    try:
        font = ImageFont.truetype("arial.ttf", FONT_SIZE)
    except OSError:
        font = ImageFont.load_default()

    def lonlat_to_sar_pixel(transformer, lon, lat):
        # Convert geographic coordinates to native SAR pixel coordinates.
        row, col = transformer.rowcol(lon, lat, op=lambda value: value)
        return float(col), float(row)

    def heading_to_sar_unit(transformer, lon, lat, heading, x0, y0, step_m=1000.0):
        # Convert aircraft heading into a unit vector in SAR image coordinates.
        heading_rad = np.deg2rad(heading)
        lat_rad = np.deg2rad(lat)
        d_lat = (step_m * np.cos(heading_rad)) / 111320.0
        d_lon = (step_m * np.sin(heading_rad)) / (111320.0 * np.cos(lat_rad))
        x1, y1 = lonlat_to_sar_pixel(transformer, lon + d_lon, lat + d_lat)
        dx, dy = x1 - x0, y1 - y0
        length = float(np.hypot(dx, dy))
        if not np.isfinite(length) or length == 0:
            return 0.0, -1.0
        return dx / length, dy / length

    final_img, transformer, scaling = load_period_image(tiff_path)
    try:
        draw = ImageDraw.Draw(final_img)

        try:
            plane_data = {}
            for row in df_filtered.itertuples():
                px, py = lonlat_to_sar_pixel(transformer, row.lon, row.lat)
                # Apply the height-induced range correction while preserving
                # the Doppler-based azimuth displacement. GPS positions are
                # retained as reference locations.
                pgx = px + row.height_range_shift_px
                pgy = py + row.doppler_offset_px
                heading_vector = heading_to_sar_unit(
                    transformer,
                    row.lon,
                    row.lat,
                    row.heading,
                    px,
                    py,
                )
                p_id = str(row.icao24)
                callsign = str(getattr(row, 'callsign', '')).strip()
                label = f'{callsign} ({p_id})' if callsign and callsign.lower() != 'nan' else p_id
                if p_id not in plane_data:
                    plane_data[p_id] = {'gps': [], 'doppler': [], 'heading_vector': heading_vector, 'label': label, 'callsign': callsign, 'points': []}
                plane_data[p_id]['gps'].append((int(px), int(py)))
                plane_data[p_id]['doppler'].append((int(pgx), int(pgy)))
                plane_data[p_id]['points'].append({'col': float(pgx), 'row': float(pgy), 'height': float(row.geoaltitude)})
        finally:
            transformer.close()

        d_ellipse = draw.ellipse
        d_line = draw.line
        d_polygon = draw.polygon
        d_text = draw.text

        records = []
        for i, (p_id, data) in enumerate(plane_data.items()):
            hue = (i * 0.137) % 1.0
            r, g, b = colorsys.hsv_to_rgb(hue, 1.0, 1.0)
            colour = (int(r*255), int(g*255), int(b*255))

            gps, dopp = data['gps'], data['doppler']
            p1_g, p2_g = gps[0], gps[-1]
            p1_d, p2_d = dopp[0], dopp[-1]

            for p in (p1_g, p2_g):
                d_ellipse([p[0]-POINT_SIZE, p[1]-POINT_SIZE, p[0]+POINT_SIZE, p[1]+POINT_SIZE], fill=colour)

            # Retrieve the SAR heading unit vector.
            ux, uy = data['heading_vector']
        
            # Compute the end point of the aircraft heading vector.
            hx = p1_g[0] + (HEADING_LEN * ux)
            hy = p1_g[1] + (HEADING_LEN * uy)

            # Direction vector in SAR image coordinates.
            sub_x = ux
            sub_y = uy

            # Unit vector perpendicular to the flight direction.
            per_x = -sub_y
            per_y = sub_x

            ex, ey = DOPPLER_WIDTH * sub_x, DOPPLER_WIDTH * sub_y
            ox, oy = DOPPLER_WIDTH * per_x, DOPPLER_WIDTH * per_y

            # FIX: Explicitly cast all calculated corners to standard Python integers
            corners = [
                (int(p1_d[0] - ex + ox), int(p1_d[1] - ey + oy)),
                (int(p2_d[0] + ex + ox), int(p2_d[1] + ey + oy)),
                (int(p2_d[0] + ex - ox), int(p2_d[1] + ey - oy)),
                (int(p1_d[0] - ex - ox), int(p1_d[1] - ey - oy))
            ]

            records.append({'icao24': p_id, 'callsign': data['callsign'], 'range_columns': [p['col'] for p in data['points']], 'doppler_rows': [p['row'] for p in data['points']], 'altitudes_m': [p['height'] for p in data['points']], 'corners_px': [list(p) for p in corners]})
            d_polygon(corners, outline=colour, width=OUTLINE_WIDTH)
            d_line([p1_g, (int(hx), int(hy))], fill=colour, width=15)
            d_text((p1_g[0] + 80, p1_g[1] - TEXT_DIS), data['label'], fill=(255, 255, 255), font=font)


        final_img.save(output_jpg, "JPEG", quality=95)
        return {'size': list(final_img.size), 'bytes': Path(output_jpg).stat().st_size, 'scaling': scaling, 'tracks': records}
    finally:
        final_img.close()


# Sentinel-1 Processing and Image Generation

Execute the complete workflow, including scene selection, aircraft-track preparation, image rendering, quality checks, and output reporting.

In [ ]:
NOTEBOOK_CODE_ID = '092bf6861afe6824b643cb500ff4387f694c11aa7cd228e0a86dfaa8b1bb817a'

def render_scene(tiff, frame, output, margin_km, xml_path=None):
    annotation = ET.parse(xml_path or Path(tiff).with_suffix('.xml')).getroot()
    for key in ('rangePixelSpacing', 'azimuthPixelSpacing'):
        if abs(float(annotation.findtext('.//imageInformation/' + key)) - 10.0) > 1e-8:
            raise ValueError('Rendering requires the original 10 m GRDH pixel grid.')
    output = Path(output)
    output.parent.mkdir(parents=True, exist_ok=True)
    temporary = output.with_name(output.name + '.part')
    details = draw_sar_image(tiff, frame, temporary, margin_km=margin_km)
    with Image.open(temporary) as image:
        if list(image.size) != details['size']:
            raise RuntimeError('Saved JPEG dimensions differ from the original image.')
        image.verify()
    temporary.replace(output)
    details['bytes'] = output.stat().st_size
    return details

def valid_output(output, signature):
    sidecar=Path(output).with_suffix('.json')
    if not Path(output).is_file() or not sidecar.is_file():return False
    try:
        record=json.loads(sidecar.read_text(encoding='utf-8'))
        if record.get('signature')!=signature:return False
        if record['bytes']!=Path(output).stat().st_size:return False
        with Image.open(output) as image:
            if list(image.size)!=record['size']:return False
            image.verify()
        return True
    except (OSError,KeyError,ValueError):return False

def main(argv=None):
    global _QUERY_CLIENT
    parser = argparse.ArgumentParser(
        description=(
            "Process Sentinel-1 scenes with aircraft height correction. "
            "Dates are inclusive and interpreted in UTC."
        )
    )
    parser.add_argument('--root',type=Path,default=ROOT,help='Base folder; old VV folders beneath it are not scanned')
    parser.add_argument('--download-dir',type=Path,help='Directory containing Sentinel-1 downloads')
    parser.add_argument('--output',type=Path,help='Image/report/cache folder')
    parser.add_argument('--from-date',default=DATE_FROM)
    parser.add_argument('--to-date',default=DATE_TO_INCLUSIVE)
    parser.add_argument('--margins-km',type=float,nargs='+',default=list(DEFAULT_MARGINS_KM))
    parser.add_argument('--opensky-user',default=os.getenv('OPENSKY_USER',''))
    parser.add_argument('--inventory',action='store_true',help='Only list scenes/missing local files; no login/download/JPEG')
    parser.add_argument('--local-only',action='store_true',help='Only scenes already local; does not claim complete date coverage')
    parser.add_argument('--offline',action='store_true',help='No network: local scenes and complete ADS-B caches only')
    parser.add_argument('--refresh-catalogue',action='store_true')
    parser.add_argument('--limit-scenes',type=int,help='Process the first N scenes for a trial run')
    args=parser.parse_args(argv)
    begin=pd.to_datetime(args.from_date,utc=True).normalize()
    stop=pd.to_datetime(args.to_date,utc=True).normalize()+pd.Timedelta(days=1)
    if stop<=begin:parser.error('The end date must not precede the start date.')
    if any(not np.isfinite(m) or m<=0 for m in args.margins_km):parser.error('Margins must be positive finite kilometres.')
    if args.limit_scenes is not None and args.limit_scenes<1:parser.error('--limit-scenes must be positive.')
    if not args.root.is_dir():parser.error('The input root directory does not exist.')
    margins=sorted(set(args.margins_km))
    output = args.output or args.root / 'Processed_Images'
    output=output.resolve();work=output/'cache'
    download_dir=(args.download_dir or args.root/'Sentinel1_Downloads').resolve()
    output.mkdir(parents=True,exist_ok=True);work.mkdir(parents=True,exist_ok=True)
    session=public_session()
    print('Checking download directory:', download_dir, flush=True)
    index=scan_local(download_dir,validate=False)
    print('Existing XML/TIFF pairs:',sum(bool(p['pair']) for p in index.values()),flush=True)
    identity={'from':begin.isoformat(),'stop_exclusive':stop.isoformat(),'bbox':BBOX,'orbit':ORBIT_DIRECTION,'product':'IW_GRDH_1S','polarisation':'VV'}
    catalog_key=hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()[:20]
    catalog_cache=work/f'catalogue_{catalog_key}.json'
    local_mode=args.local_only or args.offline
    if local_mode:
        products=local_products(index,begin,stop)
    elif catalog_cache.exists() and not args.refresh_catalogue:
        record=json.loads(catalog_cache.read_text(encoding='utf-8'))
        products=record['products']
    else:
        print('Fetching public Sentinel catalogue metadata...',flush=True)
        products=catalogue_products(session,begin,stop)
        write_json(catalog_cache,{'selection':identity,'products':products})
    inventory={'selection':identity,'local_only':bool(local_mode),'catalogue_coverage_checked':not local_mode,'products':products,'matching_products':len(products),'reusable_local_scenes':sum(bool(index.get(p['Name'],{}).get('pair')) for p in products),'missing_local':[p['Name'] for p in products if not index.get(p['Name'],{}).get('pair')],'margins_km':margins,'images_if_successful':len(products)*len(margins)}
    inventory.update(download_directory=str(download_dir),local_file_check='Matching names and nonzero sizes only; actual scene validated during image processing. Old VV directories are not searched.')
    write_json(output/'inventory.json',inventory)
    print(f"Period {args.from_date} through {args.to_date}: {len(products)} scenes; {inventory['reusable_local_scenes']} reusable, {len(inventory['missing_local'])} missing; {inventory['images_if_successful']} JPEGs for all margins.",flush=True)
    if local_mode:print('Local/offline inventory: complete coverage of the date period is NOT asserted.',flush=True)
    if args.inventory:
        session.close();return 0
    products=products[:args.limit_scenes] if args.limit_scenes else products
    _QUERY_CLIENT=OpenSkyCache(work/'adsb',args.opensky_user,args.offline)
    downloads=CopernicusDownloads(session,args.offline)
    cursor=SimpleNamespace(description=None)
    # The processing workflow applies a Doppler-based azimuth displacement
    # approximation together with a height-induced range correction.
    # The resulting geometry does not independently confirm target identity.
    fingerprint = hashlib.sha256((NOTEBOOK_CODE_ID + json.dumps({"bbox": BBOX, "polygon": polygon_wkt, "buffer": buffer, "inclination": inclination, "orbit": ORBIT_DIRECTION}, sort_keys=True)).encode()).hexdigest()
    report={'selection':identity,'local_only':bool(local_mode),'margins_km':margins,'script_sha256':fingerprint,'scenes':[]}
    fatal=None
    try:
        # Authenticate before downloading additional Sentinel-1 products.
        # Existing authentication tokens are reused when available.
        if not args.offline:
            _QUERY_CLIENT.authenticate()
        for number,product in enumerate(products,1):
            name=product['Name'];stamp=pd.to_datetime(product['ContentDate']['Start'],utc=True)
            entry={'product':name,'date':stamp.isoformat(),'versions':[]}
            report['scenes'].append(entry)
            print(f'[{number}/{len(products)}] {name}',flush=True)
            try:
                pending=[]
                for margin in margins:
                    tag=f'{margin:g}'.replace('.','p')
                    image_path=output/f'margin_{tag}km'/stamp.strftime('%Y-%m')/(name.removesuffix('.SAFE')+'.jpg')
                    signature={'product':name,'margin_km':margin,'script_sha256':fingerprint}
                    version={'margin_km':margin,'output':str(image_path)}
                    entry['versions'].append(version)
                    if valid_output(image_path,signature):
                        version['status']='existing'
                    else:pending.append((margin,image_path,signature,version))
                if not pending:
                    entry['status']='existing';continue
                scene=downloads.ensure_scene(product,index,download_dir)
                entry.update(scene)
                start,end,_=scene_times_from_tiff_name(Path(scene['tiff']))
                frame=prepare_aircraft_dataframe(cursor,start,end,Path(scene['xml']))
                entry['usable_aircraft']=int(frame.icao24.nunique()) if not frame.empty else 0
                if frame.empty and not SAVE_SCENES_WITHOUT_AIRCRAFT:
                    entry['status']='no_usable_aircraft'
                    for _,_,_,version in pending: version['status']='skipped_no_aircraft'
                    print('No usable aircraft tracks found; image generation skipped.',flush=True)
                    continue
                geometry_path=work/'geometry'/(name.removesuffix('.SAFE')+'.json')
                geometry_rows=frame.drop(columns=['geometry'],errors='ignore').to_dict('records') if not frame.empty else []
                write_json(geometry_path,{'product':name,'aircraft_tracks':geometry_rows})
                for margin,image_path,signature,version in pending:
                    print(f"  Rendering {margin:g} km; {entry['usable_aircraft']} aircraft",flush=True)
                    details=render_scene(Path(scene['tiff']),frame,image_path,margin,scene['xml'])
                    details.update(signature=signature,product=name,margin_km=margin,aircraft_count=entry['usable_aircraft'])
                    write_json(image_path.with_suffix('.json'),details)
                    version.update(status='saved',bytes=details['bytes'])
                entry['status']='saved'
            except KeyboardInterrupt:
                entry['status']='interrupted';fatal='Interrupted; completed images/caches are retained.';break
            except Exception as exc:
                entry['status']='failed';entry['error']=f'{type(exc).__name__}: {exc}'
                print('  FAILED:',entry['error'],flush=True)
                # Stop processing if authentication fails.
                # Scene-specific failures are logged and skipped.
                if type(exc).__name__ in ('TrinoAuthError','OAuth2Error') or 'login failed' in str(exc).lower():
                    fatal=entry['error'];break
            finally:
                write_json(output/'run_report.json',report)
                _SCALING.clear();gc.collect()
    except KeyboardInterrupt:
        fatal = ('Interrupted during OpenSky authentication. Restart authentication and try again.')
    except Exception as exc:
        fatal=f'{type(exc).__name__}: {exc}'
        print('Processing stopped before scene execution:',fatal,flush=True)
    finally:
        _QUERY_CLIENT.close();session.close()
    counts={s:sum(v.get('status')==s for e in report['scenes'] for v in e['versions']) for s in ('saved','existing')}
    counts['failed_scenes']=sum(e['status']=='failed' for e in report['scenes'])
    counts['scenes_attempted']=len(report['scenes']);counts['scenes_planned']=len(products)
    report['summary']=counts
    if fatal:report['stopped_reason']=fatal
    write_json(output/'run_report.json',report)
    print('Finished:',json.dumps(counts),flush=True)
    print('Images/reports:',output,flush=True)
    if fatal:print(fatal,flush=True)
    return 1 if fatal or counts['failed_scenes'] else 0

def notebook_arguments():
    args = [
        "--root", str(ROOT),
        "--output", str(OUTPUT_DIR),
        "--download-dir", str(DOWNLOAD_DIR),
        "--from-date", DATE_FROM,
        "--to-date", DATE_TO_INCLUSIVE,
        "--margins-km", *[str(m) for m in MARGINS_KM],
    ]
    if OPENSKY_USER.strip():
        args += ["--opensky-user", OPENSKY_USER.strip()]
    if LOCAL_ONLY:
        args.append("--local-only")
    if OFFLINE:
        args.append("--offline")
    if REFRESH_CATALOGUE:
        args.append("--refresh-catalogue")
    if LIMIT_SCENES is not None:
        args += ["--limit-scenes", str(LIMIT_SCENES)]
    return args


# Generate Scene Inventory

Identify all matching Sentinel-1 acquisitions, check local file availability, and estimate the number of images that will be generated.

In [ ]:
inventory_status = main(notebook_arguments() + ["--inventory"])

inventory = json.loads(
    (OUTPUT_DIR / "inventory.json").read_text(encoding="utf-8")
)

display(
    pd.DataFrame(
        [{
            "Matching Scenes": inventory["matching_products"],
            "Local XML/TIFF Pairs": inventory["reusable_local_scenes"],
            "Scenes Missing Files": len(inventory["missing_local"]),
            "Planned JPEG Images": inventory["images_if_successful"],
        }]
    )
)

# OpenSky Authentication

Authenticate with OpenSky before running the processing workflow. In offline mode, cached ADS-B data are used without browser-based login.


In [ ]:
if OFFLINE:
    print(
        "Offline mode: using only cached ADS-B data. "
        "No browser authentication will be performed."
    )
else:
    OPENSKY_USER = login_opensky(OPENSKY_USER)

# Run the Processing Workflow

Execute the complete Sentinel-1 aircraft-detection workflow and report the final processing status.

In [ ]:
run_status = main(notebook_arguments())

if run_status == 0:
    print("Processing completed successfully. Output directory:", OUTPUT_DIR)
else:
    print(
        "Processing failed or was interrupted. "
        "See run_report.json for details:",
        OUTPUT_DIR
    )